# Clean GLIMS Regions
Author: Ann Windnagel

Date: 3/10/19

Revision: 4/8/2024 to work with the latest version of GLIMS. Speicfically to deal with Region 5 (Greenlad periphery) that now includes a connectivity level.

Description:  
The glacier outlines in the GLIMS database are multitemporal, so each glacier has many entries. For the analysis to find the largest glaciers in the world, need to go with the latest glacier entry. Removes unneeded columns from the shapefiles. In addition, GLIMS also has outlines for debris cover and rock outcrops as well as the glacier outlines, so need to pull out only glacier outlines. This will make the GLIMS database smaller and easier to work with. Note: Some glaciers have more than one entry with the same time stamp. This was addressed in the clean_glims function in the wgms_scripts.py file.

## Import Packages

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import earthpy as et

# set working dir
os.chdir(os.path.join(et.io.HOME, "git/wgms-glacier-project"))

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import scripts.wgms_scripts as ws

## Clean Regions

In [2]:
# Set GLIMS version
version = '20250603'
cleaned_fp = 'data/glims/processed/glims_version_' + version + '/cleaned'
if not os.path.exists(cleaned_fp):
    print('creating cleaned directory')
    os.makedirs(cleaned_fp)

# Clean glims
region_no = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
for region in region_no:
    glims_region_fp = 'data/glims/processed/glims_version_' + version + '/glims_region_' + str(region) + '.shp'
    region_clean_fp = 'data/glims/processed/glims_version_' + version + '/cleaned/glims_region_' + \
                      str(region) + "_cleaned.shp"
    if os.path.exists(region_clean_fp) == False:
        print('Cleaning region: ' + str(region))
        glims_region = gpd.read_file(glims_region_fp)
        ws.clean_glims(glims_region, region_clean_fp, region)
    else:
        print(region_clean_fp + ' already exists')

Cleaning region: 1
Cleaning region: 2
Cleaning region: 3
Cleaning region: 4
Cleaning region: 5
Cleaning region: 6
Cleaning region: 7
Cleaning region: 8
Cleaning region: 9
Cleaning region: 10
Cleaning region: 11
Cleaning region: 12
Cleaning region: 13
Fixing G072126E38989N
2002-07-10T00:00:00
Cleaning region: 14
Cleaning region: 15
Cleaning region: 16
Cleaning region: 17
Cleaning region: 18
Cleaning region: 19


### Extra Cleaning for Region 11

Region 11 contains outlines from 1850 which are of glaciers that have melted. Need to remove these from the region 11 GLIMS database so that they are not included in this analysis.

In [3]:
# Open GLIMS Region 11 shapefile with all of the glacier outlines
glims_r11_all_glaciers_fn = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_11_cleaned.shp"
glims_r11_all_glaciers_df = gpd.read_file(glims_r11_all_glaciers_fn)
#print(len(glims_r11_all_glaciers_df))

# Search for the 1850 glaciers and get the index for each instance
glaciers_1850 = glims_r11_all_glaciers_df[glims_r11_all_glaciers_df['src_date'].str.contains("1850")]
glaciers_1850_index = glaciers_1850.index
print("number of 1850 outlines: ", len(glaciers_1850))

if len(glaciers_1850) != 0:
    # Remove the 1850 glaciers and save the new file
    glaciers_1850_removed = glims_r11_all_glaciers_df.drop(glaciers_1850_index)
    len(glaciers_1850_removed)

    # Save the newly cleaned region 11 file
    region_11_clean_fp = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_11_cleaned.shp"
    glaciers_1850_removed.to_file(driver='ESRI Shapefile', filename=region_11_clean_fp)
    print("1850 glaciers have been cleaned from region 11")
elif len(glaciers_1850) == 0:
    print("1850 glaciers have already been cleaned from region 11, no action necessary")

number of 1850 outlines:  474


### Extra Cleaning for newer versions of Region 5
Newer verions of GLIMS (v20230607, v20240603) now have a connectivity level attribute (conn_lvl). We are only interested in glaciers with connectivity of 0 or 1. Extract these and resave clean shapefile.

In [4]:
if version != '20190304':
    # Open cleaned GLIMS region 05 - Greenland Periphery
    glims_region5_fp = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_5_cleaned.shp"
    glims_region5_polygons = gpd.read_file(glims_region5_fp)
    
    # Check number of polygons in the pre-cleaned dataframe
    preclean = len(glims_region5_polygons)
    print("preclean: ", preclean)
    
    # Select glaciers that have connectivity level of 0 or 1
    glims_region5_polygons = glims_region5_polygons.loc[
        (glims_region5_polygons['conn_lvl'] == 0) | (glims_region5_polygons['conn_lvl'] == 1)]
    
    # Check the number of resulting polygons in the clean dataframe
    postclean = len(glims_region5_polygons)
    print("postclean: ", postclean)
    
    # Save new shapefile if needed
    if preclean == postclean:
        # Cleaning already done or not needed
        print("The pre-clean and post-clean polygon counts are the same. No cleaning needed.")
    elif preclean > postclean:
        # Write dataframe to shapefile
        clean_fn = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_5_cleaned.shp"
        glims_region5_polygons.to_file(driver='ESRI Shapefile', filename=clean_fn)
        print("Writing cleaned region 5 shapefile")

preclean:  22500
postclean:  21559
Writing cleaned region 5 shapefile


In [5]:
# Open the new shapefile to make sure it is okay.
if version != '20190304':
    glims_region5_fp = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_5_cleaned.shp"
    glims_region5_polygons = gpd.read_file(glims_region5_fp)
    display(glims_region5_polygons.head())
    
    print(glims_region5_polygons[['conn_lvl']].max())
    print(glims_region5_polygons[['conn_lvl']].min())

,region_no,glac_id,area,db_area,width,length,min_elev,mean_elev,max_elev,src_date,glac_name,conn_lvl,geometry
0,5,G306386E70153N,9.82728,9.77935,0.0,0.0,957.0,1411.0,1902.0,2016-08-30T00:00:00,None,0.0,"POLYGON ((-53.543659 70.161951, -53.544368 70...."
1,5,G332260E82625N,2.45500,2.45459,0.0,0.0,898.0,0.0,1157.0,1999-09-09T00:00:00,None,0.0,"POLYGON ((-27.69646 82.633252, -27.696779 82.6..."
2,5,G317513E62879N,2.48200,2.48224,0.0,0.0,366.0,0.0,1387.0,2001-09-10T00:00:00,None,0.0,"POLYGON ((-42.486297 62.888347, -42.486301 62...."
3,5,G317328E62844N,1.05000,1.04984,0.0,0.0,645.0,0.0,1411.0,2001-09-10T00:00:00,None,0.0,"POLYGON ((-42.666343 62.839719, -42.666645 62...."
4,5,G305983E69889N,1.29974,1.29360,0.0,0.0,714.0,941.0,1213.0,2016-08-30T00:00:00,None,0.0,"POLYGON ((-54.031132 69.898223, -54.030322 69...."


conn_lvl    1.0
dtype: float64
conn_lvl    0.0
dtype: float64


## Check Region 13
Region 13 has an erroneous outline of Fedchenko glacier (G072126E38989N) from 2003 with an area of only 5.15 sq km. This gets removed in wgms_scripts.clean_glims(). Need to check that it was removed by that routine. For version 20230607 and after, the date of the correct outline should be 2002-07-10.

In [10]:
# Open GLIMS Region 13 shapefile with all of the glacier outlines
if version != '20190304':
    glims_r13_all_glaciers_fn = "data/glims/processed/glims_version_" + version + "/cleaned/glims_region_13_cleaned.shp"
    glims_r13_all_glaciers_df = gpd.read_file(glims_r13_all_glaciers_fn)
    #print(len(glims_r13_all_glaciers_df))

    # Search for Fedchenko glacier GLIMS ID G072126E38989N
    glacier_fedchenko = glims_r13_all_glaciers_df[glims_r13_all_glaciers_df['glac_id'] == 'G072126E38989N']
    display(glacier_fedchenko)

,region_no,glac_id,area,db_area,width,length,min_elev,mean_elev,max_elev,src_date,glac_name,geometry
53280,13,G072126E38989N,0.0,677.653,0.0,0.0,0.0,0.0,0.0,2002-07-10T00:00:00,None,"POLYGON ((72.454178 38.793632, 72.454463 38.79..."
